In [1]:
import pandas as pd
import requests
import json

from datetime import datetime
import glob
from enum import Enum
from pathlib import Path

from aind_data_schema.core.procedures import SpecimenProcedure, SpecimenProcedureType, ImmunolabelClass, HCRSeries, Antibody, Procedures, ViralMaterial, TarsVirusIdentifiers

from aind_data_schema.models.organizations import Organization

from aind_data_schema.models.pid_names import PIDName

from aind_data_schema.models.registry import Registry

import logging


materials_sheet = pd.read_excel("./Mouse Tracker - RO injections.xlsx", sheet_name="Mouse Tracker - RO injections", header=[0], converters={})

In [2]:
log_file_name = "./logging/log_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".log"
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
# create file handler which logs even debug messages
fh = logging.FileHandler(log_file_name, "w", "utf-8")
fh.setLevel(logging.DEBUG)

# create formatter and add it to the handlers
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
fh.setFormatter(formatter)
# add the handlers to logger
logger.addHandler(fh)

In [3]:
subj_procedures = {}

download_files = False

files = glob.glob("./original_spec_files/*.json")


tars_models = {}

if download_files:
    for file in files:    
        print(file)
        subj_id = file.split("\\")[-1].split("_")[0]
        print(subj_id)
        if int(subj_id) not in materials_sheet["Primary"].tolist():
            print("not found")
            continue

        subj_row = materials_sheet.loc[materials_sheet["Primary"] == int(subj_id)]
        
        if len(subj_id) != 6:
            continue

        for val in [1,2,3]:
            tars_id = subj_row[f"Virus{val} ID"].values[0]
            print(tars_id)

            if pd.isna(tars_id):
                continue

            if tars_id not in tars_models.keys():
                request = requests.get(f"http://aind-metadata-service/tars_injection_materials/{tars_id}")

            if request.status_code == 404 or request.status_code == 500:
                print(f"{tars_id} model not found")
                continue

            print(f"code: {request.status_code}")

        
            item = request.json()

            if not item or not item['data']:
                continue

            if item['message'] == 'Valid Model.':
                tars_models[tars_id] = item['data']
            else:
                print(f"invalid model for {tars_id}")
                print(item['message'])

print(tars_models)

for key, value in tars_models.items():
    with open(f'./tars_info/{key}.json', 'w') as outfile:
        json.dump(value, outfile)

{}


In [4]:

tars_files = glob.glob("./tars_info/*.json")
tars_models = {}
for file in tars_files:
    with open(file) as json_file:
        data = json.load(json_file)
        tars_models[data['tars_identifiers']["prep_lot_number"]] = data


In [5]:
sanity_checks = []

def get_inj_materials(subj_id):
    print(materials_sheet["Primary"].tolist())
    print(type(subj_id))
    print(type(materials_sheet["Primary"].tolist()[0]))
    if int(subj_id) not in materials_sheet["Primary"].tolist():
        logging.info(f"Subject {subj_id} not found in materials sheet")
        return []
    
    materials = []

    subj_row = materials_sheet.loc[materials_sheet["Primary"] == int(subj_id)]
    logging.info(f"subj_row: {subj_row}")
    logging.info("hullo")
    for val in [1,2,3]:
        virus = subj_row[f"Virus{val}"].values[0]
        logging.info(f"virus{val}: {virus}")
        if pd.isna(virus):
            continue

        virus_id = subj_row[f"Virus{val} ID" ].values[0] # use this to look up TARS info

        titer = subj_row[f"Virus{val} Titer (GC/mL)"].values[0]
        dose = float(subj_row[f"Virus{val} Dose (GC/mouse)"].values[0])
        volume = subj_row[f"Virus{val} Volume Injected"].values[0]
        mix_volume = subj_row[f"Virus Mix Volume injected"].values[0]
        if pd.isna(mix_volume):
            mix_volume = .1
        logging.info(f"titer: {titer}")
        logging.info(f"mix vol: {mix_volume}")

        if not pd.isna(mix_volume):
            logging.info("FOUND MIX VOL: " + str(mix_volume))


        try:
            logging.info("checking titer vs dose/volume")
            logging.info(f"titer: {titer}, dose: {dose}, volume: {volume}, mix volume: {mix_volume}")
            
            original_titer = float(titer)
            titer = original_titer

            computed_titer = False
            try:
                logging.info("trying")
                if not pd.isna(volume):
                    logging.info("volume is not NA")
                    volume = float(volume.split("u")[0])
                    titer = int(dose/(volume*.001))
                    computed_titer = True
            except:
                logging.info("volume is not a number")
            
            logging.info("out")
            mix_titer = None
            if not pd.isna(mix_volume):
                mix_volume = float(mix_volume.split("u")[0])
                logging.info("mix volume is not NA")
                mix_titer = int(dose/(mix_volume*.0001))
                logging.info(f"mix titer: {mix_titer}")
            

            if not pd.isna(mix_titer) and not pd.isna(titer):
                if mix_titer != titer:
                    logging.info(f"MISSMATCH: titer output: {titer} (computed: {computed_titer}), original titer: {original_titer}, mix titer: {mix_titer}, mix vol: {mix_volume}")
            else:
                logging.info(f"something na -- mix titer: {mix_titer}, titer: {titer}")

        except:
            logging.info("something missing")


        if pd.isna(titer): # actually, do the calculation for everything
            logging.info("titer is NA")
            
            
            if pd.isna(volume):
                logging.error(f"Volume is NA for material {val} : {subj_id}, {virus}")
                continue

            if isinstance(volume, str):
                volume = float(volume.split("u")[0])
            logging.info(f"dose: {dose}, volume: {volume}")

            titer = int(dose/(volume*.001))
        else:
            titer = float(titer)

        # do some checks to see how accurate titer is to dose/volume
            
        tars = None
        if virus_id in tars_models.keys():
            tars = TarsVirusIdentifiers.model_validate(tars_models[virus_id]["tars_identifiers"])
            logging.info(f"tars for {subj_id}: {tars}")

        logging.info(f"titer: {titer}, dose: {dose}, volume: {volume}")
        new_material = ViralMaterial(
            name=virus,
            titer=titer,
            tars_identifiers=tars, 
        )

        logging.info(f"new material: {new_material}")

        materials.append(new_material)

        logging.info(f"finished virus {val}")

    return materials


In [6]:
files = glob.glob("./original_spec_files/*.json")

for file in files:
    with open(file) as f:
        data = json.load(f)
        print(data)
        original_procedure = Procedures.model_construct(**data)

    print(original_procedure)

    subj = original_procedure.subject_id

    for surgery in original_procedure.subject_procedures:
        print(surgery)
        if "protocol_id" not in surgery.keys():
            surgery["protocol_id"] = "dx.doi.org/10.17504/protocols.io.kqdg392o7g25/v1"
            logging.info(f"adding surgery protocol id for subject {subj}")
        elif surgery["protocol_id"] == "unknown":
            logging.info(f"replacing surgery protocol id for subject {subj}")
            surgery["protocol_id"] = "dx.doi.org/10.17504/protocols.io.kqdg392o7g25/v1"
        for subj_procedure in surgery["procedures"]:
            logging.info(f"checking procedure {subj_procedure} for subject {subj}")
            if subj_procedure["procedure_type"] == "Perfusion":
                if "protocol_id" not in subj_procedure.keys():
                    logging.info(f"adding perfusion protocol id for subject {subj}")
                    subj_procedure["protocol_id"] = "dx.doi.org/10.17504/protocols.io.bg5vjy66"
                    
                elif subj_procedure["protocol_id"] == "unknown":
                    logging.info(f"replacing perfusion protocol id for subject {subj}")
                    subj_procedure["protocol_id"] = "dx.doi.org/10.17504/protocols.io.bg5vjy66"

            if subj_procedure["procedure_type"] == "Retro-orbital injection":
                logging.info(f"checking retro-orbital injection for subject {subj}")
                materials = get_inj_materials(subj)
                logging.info(f"materials for subject {subj}: {materials}")

                subj_procedure["injection_materials"] = materials
                
                

    original_procedure.write_standard_file(
        output_directory=Path("original_plus_materials"),
        prefix=subj
    )

    
print(sanity_checks)


    # titer = dose / volume, with volume in ml (gc/ml) (translate to ml)

    # perhaps put vehicle in notes field of surgery?

    # for value in [1,2,3]:


{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.11.5', 'subject_id': '576404', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2021-07-12', 'experimenter_full_name': '28908', 'iacuc_protocol': '1806', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['576404']}], 'notes': None}], 'specimen_procedures': [], 'notes': None}
describedBy='https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py' schema_version='0.11.5' subject_id='576404' subject_procedures=[{'procedure_type': 'Surgery', 'start_date': '2021-07-12', 'experimenter_full_name': '28908', 'iacuc_protocol': '1806', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthes

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, def

{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.11.5', 'subject_id': '620631', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2022-04-01', 'experimenter_full_name': '13040', 'iacuc_protocol': '2104', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['620631']}], 'notes': None}], 'specimen_procedures': [], 'notes': None}
describedBy='https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py' schema_version='0.11.5' subject_id='620631' subject_procedures=[{'procedure_type': 'Surgery', 'start_date': '2022-04-01', 'experimenter_full_name': '13040', 'iacuc_protocol': '2104', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthes

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-pac

{'procedure_type': 'Surgery', 'start_date': '2022-10-28', 'experimenter_full_name': '13040', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['648700']}], 'notes': None}
{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.11.5', 'subject_id': '648858', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2022-09-26', 'experimenter_full_name': '13040', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'injection_materials': [], 'recovery_time': None, 'recovery_time_unit': 'minute', 'injection_duration': None, 'injection_duration_unit': 'minute', 'instrument_id': None, 'procedure_type':

...

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, def

{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.11.5', 'subject_id': '653431', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2023-01-12', 'experimenter_full_name': '13040', 'iacuc_protocol': '2002', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['653431']}], 'notes': None}], 'specimen_procedures': [], 'notes': None}
describedBy='https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py' schema_version='0.11.5' subject_id='653431' subject_procedures=[{'procedure_type': 'Surgery', 'start_date': '2023-01-12', 'experimenter_full_name': '13040', 'iacuc_protocol': '2002', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthes

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-pac

{'procedure_type': 'Surgery', 'start_date': '2023-02-21', 'experimenter_full_name': '13040', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'procedure_type': 'Perfusion', 'output_specimen_ids': ['660949']}], 'notes': None}
{'describedBy': 'https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/procedures.py', 'schema_version': '0.11.5', 'subject_id': '660950', 'subject_procedures': [{'procedure_type': 'Surgery', 'start_date': '2023-01-10', 'experimenter_full_name': '13040', 'iacuc_protocol': '2109', 'animal_weight_prior': None, 'animal_weight_post': None, 'weight_unit': 'gram', 'anaesthesia': None, 'workstation_id': None, 'procedures': [{'injection_materials': [], 'recovery_time': None, 'recovery_time_unit': 'minute', 'injection_duration': None, 'injection_duration_unit': 'minute', 'instrument_id': None, 'procedure_type':

c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-packages\pydantic\main.py:352: UserWarning: Pydantic serializer warnings:
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  Expected `Union[Surgery, TrainingProtocol, WaterRestriction, definition-ref]` but got `dict` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
c:\Users\mae.moninghoff\AppData\Local\miniconda3\envs\ainds\Lib\site-pac